# Phase-3: Statistical Validation of Features (London-final-light)

**Goal:** Prove each feature deserves to be in the model before building the risk index.

**london-final-light excludes `stop_search_rate` and `seasonal_volatility` by design** (ethical concerns around stop & search bias; group decision to drop seasonal volatility). Those two columns are therefore not entered into feature selection.

- **3a. Feature selection:** Spearman correlation + VIF (multicollinearity)
- **3b. Seasonality context:** Kruskal-Wallis + Coefficient of Variation + STL (London-wide, diagnostic only)
- **3c. Spatial validation:** Moran's I (spatial autocorrelation)

**Reads:** `outputs/phase2/` (shared) · **Writes:** `london-final-light/outputs/phase3/`

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

from scipy import stats
from statsmodels.tsa.seasonal import STL
import libpysal.weights as lps_weights
import esda

print('All imports OK')

In [ ]:
# Loading Dataset
BASE = Path('Dataset path')
P2   = BASE / 'london_midterm' / 'outputs' / 'phase2'                 # shared with main London model
P1   = BASE / 'london_midterm' / 'outputs' / 'phase1'                 # shared with main London model
OUT  = BASE / 'london-final-light' / 'outputs' / 'phase3'
OUT.mkdir(parents=True, exist_ok=True)

SHP_DIR = BASE / 'data' / 'LB_shp'

# Features to validate (crime_count is the outcome variable).
# stop_search_rate and seasonal_volatility are EXCLUDED from london-final-light.
FEATURES = [
    'severity_weighted_count',
    'resolution_rate',
    'imd_rank',
    'income_rank',
    'employment_rank',
    'ntl_mean_radiance',
]

SPEARMAN_THRESHOLD = 0.1   # |r| < 0.1 -> drop
VIF_THRESHOLD      = 5.0   # VIF > 5  -> drop (multicollinearity)

report_lines = []
print('Config loaded. Output folder:', OUT)
print('Excluded by design: stop_search_rate, seasonal_volatility, total_footfall')
print('Added: ntl_mean_radiance (VIIRS night-time radiance)')

## Load Data

In [ ]:
fm      = pd.read_parquet(P2 / 'phase2_feature_matrix.parquet')
# London-final-light: add VIIRS night-time radiance, footfall dropped
ntl = pd.read_parquet(BASE / 'data' / 'lsoa_nightlights_england.parquet')[['lsoa21cd', 'ntl_mean_radiance']]
fm = fm.merge(ntl, on='lsoa21cd', how='left')
fm['ntl_mean_radiance'] = fm['ntl_mean_radiance'].fillna(fm['ntl_mean_radiance'].median())
print(f"NTL radiance merged: matched={fm['ntl_mean_radiance'].notna().sum()}, null={fm['ntl_mean_radiance'].isna().sum()}")
monthly = pd.read_parquet(P2 / 'phase2_monthly_crime_counts.parquet')
crimes  = pd.read_parquet(P1 / 'phase1_crimes_london.parquet')

# Load London LSOA shapefile (borough shapefiles) for Moran's I
gdfs   = [gpd.read_file(f) for f in sorted(SHP_DIR.glob('*.shp'))]
london = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs=gdfs[0].crs)
london = london[['lsoa21cd', 'geometry']].copy()

LONDON_LSOAS = set(london['lsoa21cd'])
crimes = crimes[crimes['LSOA code'].isin(LONDON_LSOAS)].copy()
crimes = crimes.rename(columns={'LSOA code': 'lsoa21cd'})

print(f'Feature matrix: {fm.shape}')
print(f'Monthly counts: {monthly.shape}')
print(f'Crimes (London): {crimes.shape}')

In [ ]:
# Handle nulls: resolution_rate — LSOAs with zero crimes. Fill with London mean.
mean_res = fm['resolution_rate'].mean()
fm['resolution_rate'] = fm['resolution_rate'].fillna(mean_res)
print(f'resolution_rate nulls filled with London mean: {mean_res:.2f}%')
print(f'Remaining nulls: {fm[FEATURES + ["crime_count"]].isnull().sum().to_dict()}')

## 3a. Feature Selection
### Step 1: Spearman Correlation
Correlate each feature with `crime_count`. Drop features with |r| < 0.1. Spearman is used because crime data is right-skewed and not normally distributed.

In [ ]:
print('--- Spearman Correlation with crime_count ---\n')
spearman_results = []
for feat in FEATURES:
    r, p = stats.spearmanr(fm[feat], fm['crime_count'])
    passed = abs(r) >= SPEARMAN_THRESHOLD
    spearman_results.append({'feature': feat, 'spearman_r': round(r, 4), 'p_value': round(p, 6), 'pass': passed})
    print(f'  {feat:<30}  r={r:+.4f}  p={p:.2e}  [{"PASS" if passed else "FAIL"}]')

spearman_df = pd.DataFrame(spearman_results).sort_values('spearman_r', key=abs, ascending=False)
features_after_spearman = spearman_df[spearman_df['pass']]['feature'].tolist()
dropped_spearman = spearman_df[~spearman_df['pass']]['feature'].tolist()

print(f'\nFeatures passing Spearman (|r| >= {SPEARMAN_THRESHOLD}): {features_after_spearman}')
print(f'Dropped: {dropped_spearman}')

report_lines.append('=== 3a: SPEARMAN CORRELATION ===')
for _, row in spearman_df.iterrows():
    report_lines.append(f"  {row['feature']:<30}  r={row['spearman_r']:+.4f}  p={row['p_value']:.2e}  {'PASS' if row['pass'] else 'FAIL'}")
report_lines.append(f'Dropped: {dropped_spearman}\n')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
colours = ['#1D9E75' if r['pass'] else '#E24B4A' for _, r in spearman_df.iterrows()]
bars = ax.barh(spearman_df['feature'], spearman_df['spearman_r'].abs(), color=colours, edgecolor='white')
ax.axvline(SPEARMAN_THRESHOLD, color='black', linestyle='--', linewidth=1.2, label=f'Threshold |r|={SPEARMAN_THRESHOLD}')
for bar, sign in zip(bars, spearman_df['spearman_r']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2, f'{sign:+.3f}', va='center', fontsize=9)
ax.set_xlabel('|Spearman r|', fontsize=11)
ax.set_title('Spearman correlation with crime_count — london-final-light (green=pass, red=fail)', fontsize=12)
ax.legend(fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUT / '3a_spearman_correlation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 3a_spearman_correlation.png')

### Step 2: VIF Test (multicollinearity)
Iteratively drop the feature with the highest VIF while any VIF > 5. IMD/income/employment ranks are likely collinear — only the sub-score with lowest VIF is kept.

In [ ]:
def compute_vif(df, features):
    """VIF_i = 1 / (1 - R^2_i), computed with numpy only."""
    X = df[features].dropna().copy()
    X = (X - X.mean()) / X.std()
    X_vals = X.values
    n = X_vals.shape[0]
    vif_values = []
    for i in range(len(features)):
        y = X_vals[:, i]
        X_others = np.delete(X_vals, i, axis=1)
        X_others = np.column_stack([np.ones(n), X_others])
        beta, _, _, _ = np.linalg.lstsq(X_others, y, rcond=None)
        y_hat = X_others @ beta
        ss_res = np.sum((y - y_hat) ** 2)
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0
        vif_values.append(1 / (1 - r2) if r2 < 1 else np.inf)
    return pd.DataFrame({'feature': features, 'VIF': vif_values}).sort_values('VIF', ascending=False)

print('--- VIF Test (iterative) ---\n')
report_lines.append('=== 3a: VIF TEST ===')

current_features = features_after_spearman.copy()
dropped_vif = []
iteration = 0
while True:
    iteration += 1
    vif_df = compute_vif(fm, current_features)
    print(f'Iteration {iteration}:')
    print(vif_df.to_string(index=False))
    worst = vif_df.iloc[0]
    if worst['VIF'] > VIF_THRESHOLD:
        print(f"  -> Dropping '{worst['feature']}' (VIF={worst['VIF']:.2f})\n")
        dropped_vif.append(worst['feature'])
        current_features.remove(worst['feature'])
        report_lines.append(f"  Iteration {iteration}: drop '{worst['feature']}' (VIF={worst['VIF']:.2f})")
    else:
        print(f'  -> All VIF <= {VIF_THRESHOLD}. Stopping.')
        break

features_validated = current_features
print(f'\nValidated features after VIF: {features_validated}')
print(f'Dropped by VIF: {dropped_vif}')
report_lines.append(f'Dropped by VIF: {dropped_vif}')
report_lines.append(f'Validated features: {features_validated}\n')

## 3b. Seasonality Context (diagnostic)
Note: `seasonal_volatility` is **not** a feature in london-final-light. These tests are retained as general evidence of crime seasonality, but no seasonal feature is built from them.

### Step 3: Kruskal-Wallis Test

In [ ]:
london_monthly = monthly.groupby('Month')['monthly_crime_count'].sum().reset_index()
london_monthly['month_num'] = pd.to_datetime(london_monthly['Month']).dt.month

groups = [grp['monthly_crime_count'].values for _, grp in london_monthly.groupby('month_num')]
stat, p_kw = stats.kruskal(*groups)

print('--- Kruskal-Wallis Test ---')
print(f'  H-statistic: {stat:.4f}')
print(f'  p-value:     {p_kw:.4e}')
print(f'  Result:      {"SIGNIFICANT — seasonality present" if p_kw < 0.05 else "NOT significant"}')

report_lines.append('=== 3b: KRUSKAL-WALLIS TEST ===')
report_lines.append(f'  H={stat:.4f}, p={p_kw:.4e}')
report_lines.append(f'  Result: {"Seasonality present (p<0.05)" if p_kw < 0.05 else "Not significant"}\n')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pd.to_datetime(london_monthly['Month']), london_monthly['monthly_crime_count'],
        color='#378ADD', linewidth=2, marker='o', markersize=4)
ax.fill_between(pd.to_datetime(london_monthly['Month']), london_monthly['monthly_crime_count'], alpha=0.15, color='#378ADD')
ax.set_title(f'London monthly crime count (Kruskal-Wallis p={p_kw:.2e})', fontsize=12)
ax.set_ylabel('Total crimes', fontsize=10)
ax.spines[['top','right']].set_visible(False)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.savefig(OUT / '3b_monthly_crime_trend.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 3b_monthly_crime_trend.png')

### Step 4: Coefficient of Variation (CV)

In [ ]:
monthly_by_type = crimes.groupby(['Month', 'Crime type']).size().reset_index(name='count')
cv_df = monthly_by_type.groupby('Crime type')['count'].agg(mean='mean', std='std').reset_index()
cv_df['cv'] = (cv_df['std'] / cv_df['mean'] * 100).round(2)
cv_df = cv_df.sort_values('cv', ascending=False)

print('--- Coefficient of Variation by Crime Type ---')
print(cv_df[['Crime type', 'mean', 'std', 'cv']].to_string(index=False))

report_lines.append('=== 3b: COEFFICIENT OF VARIATION ===')
for _, row in cv_df.iterrows():
    report_lines.append(f"  {row['Crime type']:<35}  CV={row['cv']:.1f}%")
report_lines.append('')

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(cv_df['Crime type'], cv_df['cv'], color='#534AB7', edgecolor='white')
for i, v in enumerate(cv_df['cv']):
    ax.text(v + 0.2, i, f'{v:.1f}%', va='center', fontsize=9)
ax.set_xlabel('Coefficient of Variation (%)', fontsize=11)
ax.set_title('Seasonal volatility by crime type — London (CV = std/mean x 100)', fontsize=12)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig(OUT / '3b_cv_by_crime_type.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 3b_cv_by_crime_type.png')

### Step 5: STL Decomposition (London-wide, diagnostic)

In [ ]:
london_series = london_monthly.set_index('Month')['monthly_crime_count'].sort_index()
london_series.index = pd.to_datetime(london_series.index)

stl = STL(london_series, period=12, robust=True)
result = stl.fit()

fig, axes = plt.subplots(4, 1, figsize=(13, 9), sharex=True)
components = [
    (london_series, 'Observed', '#378ADD'),
    (result.trend,  'Trend',    '#1D9E75'),
    (result.seasonal, 'Seasonal', '#D85A30'),
    (result.resid,  'Residual', '#B4B2A9'),
]
for ax, (data, label, colour) in zip(axes, components):
    ax.plot(data.index, data.values, color=colour, linewidth=1.8)
    ax.set_ylabel(label, fontsize=10)
    ax.spines[['top','right']].set_visible(False)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].set_title('STL Decomposition — London-wide monthly crime (period=12)', fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(OUT / '3b_stl_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 3b_stl_decomposition.png')

seasonal_strength = 1 - (result.resid.var() / (result.seasonal + result.resid).var())
print(f'\nSeasonal strength: {seasonal_strength:.4f}  (>0.64 = strong, Wang et al. 2006)')
report_lines.append('=== 3b: STL DECOMPOSITION ===')
report_lines.append(f'  Seasonal strength: {seasonal_strength:.4f}')
report_lines.append(f'  Note: seasonal_volatility is NOT used as a feature in london-final-light.\n')

## 3c. Spatial Validation
### Step 6: Moran's I
Tests for spatial autocorrelation in crime counts using LSOA Queen-contiguity weights. A significant positive Moran's I justifies the spatial dimension of the risk index.

In [ ]:
london_geo = london.merge(fm[['lsoa21cd', 'crime_count']], on='lsoa21cd', how='left')
london_geo['crime_count'] = london_geo['crime_count'].fillna(0)

print('Building Queen contiguity weights...')
w = lps_weights.Queen.from_dataframe(london_geo, silence_warnings=True)
w.transform = 'r'
print(f'  Observations: {w.n}')
print(f'  Mean neighbours: {w.mean_neighbors:.2f}')

moran = esda.Moran(london_geo['crime_count'], w)
print(f"\n--- Moran's I ---")
print(f'  I:       {moran.I:.6f}')
print(f'  E[I]:    {moran.EI:.6f}')
print(f'  p-value: {moran.p_sim:.4f}')
print(f'  z-score: {moran.z_sim:.4f}')
print(f"  Result:  {'SIGNIFICANT spatial clustering confirmed' if moran.p_sim < 0.05 else 'Not significant'}")

report_lines.append("=== 3c: MORAN'S I ===")
report_lines.append(f'  I={moran.I:.6f}, E[I]={moran.EI:.6f}, p={moran.p_sim:.4f}, z={moran.z_sim:.4f}')
report_lines.append(f"  Result: {'Spatial clustering confirmed (p<0.05)' if moran.p_sim < 0.05 else 'Not significant'}\n")

In [ ]:
# Moran scatter plot (splot optional)
try:
    from splot.esda import moran_scatterplot
    fig, ax = plt.subplots(figsize=(7, 6))
    moran_scatterplot(moran, ax=ax, aspect_equal=False)
    ax.set_title(f"Moran's I = {moran.I:.4f}  (p = {moran.p_sim:.4f}) — london-final-light", fontsize=12)
    ax.spines[['top','right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(OUT / '3c_morans_i_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: 3c_morans_i_scatter.png')
except ImportError:
    print('splot not installed — skipping Moran scatter plot')

## Save Outputs

In [ ]:
validated = fm[['lsoa21cd', 'lsoa21nm', 'lad22nm', 'crime_count'] + features_validated].copy()
validated.to_parquet(OUT / 'phase3_validated_features.parquet', index=False)
print(f'Saved phase3_validated_features.parquet — {validated.shape}')
print(f'Columns: {list(validated.columns)}')

In [ ]:
report_lines.insert(0, 'CBL-16 Phase 3 — Statistical Validation Report (london-final-light)')
report_lines.insert(1, '=' * 55)
report_lines.insert(2, f'Features entering validation: {FEATURES}')
report_lines.insert(3, 'Excluded by design: stop_search_rate, seasonal_volatility')
report_lines.insert(4, f'Features validated (final):   {features_validated}')
report_lines.insert(5, '=' * 55)
report_lines.insert(6, '')

report_text = '\n'.join(report_lines)
with open(OUT / 'phase3_validation_report.txt', 'w', encoding='utf-8') as f:
    f.write(report_text)
print(report_text)

In [ ]:
print('=' * 55)
print('PHASE 3 SUMMARY (london-final-light)')
print('=' * 55)
print(f'Features entering:          {len(FEATURES)}')
print(f'Dropped by Spearman:        {len(dropped_spearman)} — {dropped_spearman}')
print(f'Dropped by VIF:             {len(dropped_vif)} — {dropped_vif}')
print(f'Features validated (final): {len(features_validated)} — {features_validated}')
print(f'Kruskal-Wallis p-value:     {p_kw:.2e}')
print(f"Moran's I:                  {moran.I:.4f} (p={moran.p_sim:.4f})")
print('=' * 55)
print(f'Outputs saved to: {OUT}')